<a href="https://colab.research.google.com/github/ralph-Jung/AI-Class/blob/main/4%EC%A3%BC%EC%B0%A8/10_5_wine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# 1. 데이터 불러오기
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/wine.csv")

print(df.head())
print(df.shape)


# 2. 입력 / 정답 분리
# 이 문제는 Outcome 을 통해서 이진 분류 하는 거임
# 그렇기에 마지막 출력층에 활성화 함수로 0과 1 사이의 값이 나오도록 하는 시그모이드 함수를 사용한다
X = df.drop("Wine", axis=1)
y = df["Wine"].values-1


# 3. 학습 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 5. 모델 생성 (입력 8 -> 은닉층 32 -> 은닉층 16 -> 출력 1)
# 첫번째 레이어 : 뉴런 32 개 , 활성화 함수 : relu
# 두번째 레이어 : 뉴런 16개 , 활성화 함수 : relu
# 세번째 레이어 : 뉴런 1개 , 활성화 함수 : sigmoid
# 출력층의 활성화 함수는 해당 문제가 이진 분류 , 다중 분류 , 회귀냐 에 따라서 달라진다
# 얘는 입력하는 컬럼의 개수가 13개 이기 때문에 shape=(13,) 으로 변경해줘야하며
# shape=(13,)는 shape 은 튜플을 받아야 하는데 (13) 이라고만 하면 튜플로 인식을 못하기 때문에 shape=(13,)로 샘플 하나가 숫자 13개짜리 1차원 데이터
# 출력층도 이진 분류가 아닌 다중 분류이고 단일 레이블 이기 때문에 softmax 함수를 사용한다
# 다중 분류에 다중 레이블이면 sigmoid 를 써야한다. 왜냐하면 softmax 는 클래스 개수만큼 확률을 낸다 그리고 전체 확률의 값의 합이 1이 된다 . 즉 , 와인의 품종이 3개니까 해당 데이터가 3개의 품종 중 어디에 속하는지를 [0.02, 0.97, 0.01] 이런식으로 확률로 제시해주고 이 중에 가장 높은 확률을 가진 품종 1이다. 라는 결론이 나온다
# 하지만 다중 레이블 이면 해당 데이터가 여러개의 정답이 가능할 수 있다 . 와인 한 병이 "달콤하면서 산미도 강할" 수 있으니까 [달콤함, 산미, 숙성] 라고 할 때 [0.91, 0.88, 0.12] 의 결과가 나오고 0.5이상인 것만 골랐을 때 [1, 1, 0] 이런식으로 달콤함, 산미 강함 모두 적용될 수도 있으니 이때는 3개의 확률의 합이 1이 되면 안되겠지? 그렇기 때문에 이때는 softmax를 쓰면 안되는 것이다
# 그래서 이런 경우는 sigmoid 를 사용해서 가중합 z를 0~1 사이 값으로 바꾸기만 하고 ( 이런식 0.91, 0.12 같은 확률 ) 거기서 역할을 끝내고 이 중에 0.5 이상만 고르도록 우리가 설정하는 것이다.
# 뉴런 3개가 각각 가중합 z를 내고, softmax가 그 3개를 한꺼번에 묶어서 합이 1이 되는 확률로 바꿔주기 때문에 마지막에 출력층의 뉴런 수를 3개로 수정
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(13,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])


# 6. 모델 설정 (한 번만)
# 다중 분류 이기 때문에 손실 함수도 기존의 binary_crossentropy 에서 categorical_crossentropy 로 변경
# 근데 categorical_crossentropy 도 정답이 아님 왜냐하면 내가 정답인 y 데이터를 원-핫 인코딩을 하지 않았기 때문이다.
# 지금 y 값은 정수 값인데 원-핫 인코딩은 품종의 수 만큼의 배열 길이로 가진 [1, 0, 0] 이런식으로 바꾸는 거임
# 만약에 이 문제에서 wine 은 총 3가지 품종이니 저렇게 배열 길이 3으로 된걸로 바꾸면 출력층에서 이런식으로 [0.91, 0.88, 0.12] 결과가 나왔을 때 y 배열에서 1이 들어있는 0.91 만 남기고 나머지는 버리는 방법이다
# 즉 , 정답과 확률 값을 비교하려면 둘이 배열의 길이가 같든 , 형식이 같아야한다
# 근데 내가 지금 가지고 있는 y 안에는 원-핫 인코딩 된게 아니라 정수값 자체가 한개 들어있는거임 . 품종에 따라 1 ,2 ,3 이런식으로 근데 출력층 결과가 [0.91, 0.88, 0.12] 이렇게 나오면 내가 가진 정답 y에는 정수 밖에 없는데 저 출력층 3개의 결과중에서 뭐를 쓰라는 거지 이런 문제가 발생함
# 그래서 이렇게 y 값을 원-핫 인코딩을 안할거면 손실 함수를 sparse_categorical_crossentropy 로 변경을 시켜준다. 그러면 이 손실함수는 y에 있는 정답 정수를 출력층의 배열 의 인덱스로 본다. 그래서 y에 0 이라는 값은 출력층 배열의 0번 인덱스의 값 인 0.91을 남기고 나머지는 버리는 것이다
# 그래서 내가 제일 앞에서 y 에서 -1 을 한것이 품종은 1 , 2 , 3 인데 배열의 인덱스는 0 , 1 , 2 이런식으로 되니까 이걸 맞춰주기 위해서 y 에서 1을 뺀 것이다
# 왜냐하면 처음에 내가 tf.keras.layers.Dense(1, activation="softmax") 로 했을 때는 categorical_crossentropy 여도 출력층에서 결과가 1개 밖에 안나오니까
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])


# 7. 모델 구조 출력
model.summary()


# 8. 모델 학습 (검증 손실이 10 에포크 동안 좋아지지 않으면 멈추고 최적 가중치로 복원)
# EarlyStopping은 val_loss 하나만 지켜보고 한 애폭 동안 가장 작은 손실을 내면 그 가중치를 기억해 두고 그 이후로 10 애폭 동안 손실이 최적일 때 보다 낮아지지 않으면 중단하고 해당 가중치로 되돌리고 학습을 중단함

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

# 에포크마다의 손실과 정확도를 기록해 둔 것
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early],
    verbose=1,
)


# 9. 테스트 데이터 예측
# 8번에서 나온 최적의 가중치를 가진 모델을 바탕으로 test 데이터를 넣어서 예측을 진행한다
# 0.5 기준은 sigmoid(이진 분류, 다중 레이블)에서 쓰는 방식
# softmax는 세 확률 중 가장 큰 것의 위치를 골라야하므로 argmax 를 이용
y_prob = model.predict(X_test)
# y_pred = (y_prob >= 0.5).astype(int).flatten()
y_pred = np.argmax(y_prob, axis=1)


# 10. 성능 평가
# y_test 에는 실제 정답값 , y_pred 에는 예측한 결과를 넣고 이런식으로 전체 중에 몇 개를 제대로 예측했는지 계산하는 것이다
# y_test = [1, 0, 1, 1, 0]   # 실제 정답
# y_pred = [1, 0, 0, 1, 1]   # 모델 예측 (0.5 기준으로 자른 값)
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy =", accuracy)
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

   Wine  Alcohol  Malic.acid   Ash   Acl   Mg  Phenols  Flavanoids  \
0     1    14.23        1.71  2.43  15.6  127     2.80        3.06   
1     1    13.20        1.78  2.14  11.2  100     2.65        2.76   
2     1    13.16        2.36  2.67  18.6  101     2.80        3.24   
3     1    14.37        1.95  2.50  16.8  113     3.85        3.49   
4     1    13.24        2.59  2.87  21.0  118     2.80        2.69   

   Nonflavanoid.phenols  Proanth  Color.int   Hue    OD  Proline  
0                  0.28     2.29       5.64  1.04  3.92     1065  
1                  0.26     1.28       4.38  1.05  3.40     1050  
2                  0.30     2.81       5.68  1.03  3.17     1185  
3                  0.24     2.18       7.80  0.86  3.45     1480  
4                  0.39     1.82       4.32  1.04  2.93      735  
(178, 14)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_27 (Dense)                │ (None, 32)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027 (4.01 KB)

 Trainable params: 1,027 (4.01 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.1504 - loss: 1.3307 - val_accuracy: 0.3103 - val_loss: 1.1884
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2389 - loss: 1.2542 - val_accuracy: 0.3448 - val_loss: 1.1071
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3097 - loss: 1.1900 - val_accuracy: 0.5862 - val_loss: 1.0327
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3805 - loss: 1.1317 - val_accuracy: 0.6552 - val_loss: 0.9638
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4513 - loss: 1.0781 - val_accuracy: 0.7241 - val_loss: 0.9021
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5221 - loss: 1.0279 - val_accuracy: 0.7586 - val_loss: 0.8462
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5487 - loss: 0.9822 - val_accuracy: 0.7931 - val_loss: 0.7942
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5664 - loss: 0.9407 - val_accuracy: 0.8276 - val_loss: